<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/ultrafiltration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***UF/MF calculation***

In [ ]:
import math as mh
import pandas as pd

def ultrafiltration(lanes,Partable2,dailyflow,Treatment):
    # UF or MF design
    # This code calculates the dimensions of the UF/MF  stage in a water treatment plant.
    #treatmenttechnologies2 = treatmenttechnologies.astype(str)

    # Input parameters
    flowRate = max(dailyflow)/24  # m3/h
    print("The daily flow rate in m3/h is",flowRate)

    #Set number of units
    units=float(input("Enter the number of units per lane(minimum 2 units): "))
    while True:
        try:
            if units >= 1:
                print("The total number of UF units in the DWTP is",units*lanes)
                break
            else:
                print("One unit per lane is required")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Nominal flow
    flowlane=flowRate/lanes # m3/h
    flowunit=flowRate/(lanes*(units-1)) # m3/h
    flow_rate=flowunit*24 #m3/day

    # Calculate permeate flow rates
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "uf").any():
      uf_recovery=float(input("Set the UF recovery rate (estimated between 0.7-0.9):"))
      uf_permeate_flow = flow_rate * uf_recovery
      print("The UF permeate flow rate (m3/day) is",uf_permeate_flow)
      Jc=int(input("The Flux Rate of the UF system is (30-50 L/m2*h):"))
    else:
      uf_recovery=float(input("Set the UF recovery rate (estimated between 0.8-0.9):"))
      uf_permeate_flow = flow_rate * uf_recovery
      print("The MF permeate flow rate (m3/day) is",uf_permeate_flow)
      Jc=int(input("The Flux Rate of the MF system is (50-80 L/m2*h):"))

    # Calculate number of membranes
    uf_area_oveall=uf_permeate_flow*41.7/Jc
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "uf").any():
      uf_area_per_membrane=50.2
      print("The selected membrane for UF is the Dupont IntegraTec Module SFP-2860 by DuPont.")
      print("The selected MF membrane has a length A of 1.9 meters, a width of 0.225 meters and feed pipe of 20 mm.The effective area is 50.2m2")
      l=1.9
      d=0.23
    else:
      uf_area_per_membrane=31.6
      print("The selected membrane for MF is the Microza UNA-620A by Pall Microza MF.")
      print("The selected MF membrane has a length A of 2.4 meters, a width of 0.221 meters and feed pipe of 20 mm. The effective area is 31.6m2")
      l=2.4
      d=0.22
    uf_membranes = mh.ceil(uf_area_oveall / uf_area_per_membrane)

    # Calculate energy consumption (kWh/day)
    # 1 bar = 10.2 mH20
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "uf").any():
      print("The operating pressure of the UF membranes is 5 bar")
      operating_pressure=5
    else:
      print("The operating pressure of the MF membranes is 3 bar")
      operating_pressure=3

    pump_efficiency=0.75
    uf_power = (uf_permeate_flow *9.81) * (operating_pressure * 10.2) / (86400 * pump_efficiency)  # kW
    Punit=uf_power
    Puf=Punit*lanes*(units-1)
    uf_energy_consumption = round(uf_power * 24)  # kWh/day
    print("The total energy consumption per unit per day (kWh) is",uf_energy_consumption)
    ufenergy=round(uf_energy_consumption*(units-1)*lanes)
    print("The total energy consumption per day (kWh) is",ufenergy)

    #Set up the membranes per vessel
    membranes_per_vessel=int(input("Set the number of membranes per vessel (6-8):"))
    uf_no_vesels=mh.ceil(uf_membranes/membranes_per_vessel)

    # Membrane characteristics
    print("No of  Membranes:", uf_membranes)
    print("Membrane Area:", uf_area_per_membrane, "m2")
    print("Permeate Flow:", uf_permeate_flow, "m3/day")
    print("Total Energy Consumption per Day:", ufenergy, "kWh/day")
    print("Number of vessels in the process:", uf_no_vesels)

    # Estimate concentration polarization factor (simplified) - This requires more detailed modeling
    membrane_orientation=input("Enter the membrane orientation (vertical or horizontal):")
    if membrane_orientation == 'vertical':
        concentration_polarization_factor = 1.2  # Example value, adjust based on actual flow conditions
    elif membrane_orientation == 'horizontal':
        concentration_polarization_factor = 1.5  # Example value, adjust based on actual flow conditions
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")

    # UF volume
    if membrane_orientation == 'vertical':
       uf_no_vesels_area=1.1*(0.3*((membranes_per_vessel*d)+((membranes_per_vessel-1)*0.2)))*uf_no_vesels
       ufunit=uf_no_vesels_area
       pump_unit=round(1.2*ufunit)
       print("pump area:",pump_unit)
       ufarea=round((ufunit+pump_unit)*units*lanes)
       print("The total area of the Ultrafiltration process (m2) is:",ufarea)
    elif membrane_orientation == 'horizontal':
       uf_no_vesels_area=1.1*(l*((membranes_per_vessel*d)+((membranes_per_vessel-1)*0.2)))
       ufunit=uf_no_vesels_area
       pump_unit=round(1.2*ufunit)
       print("pump area:",pump_unit)
       ufarea=round((ufunit+pump_unit)*units*lanes)
       print("The total area of the Ultrafiltration process (m2) is:",ufarea)
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")

    return ufarea, ufenergy
